# Predicting the Bechdel Test from Movie Metadata

**Machine Learning Foundations — Group Project**  
**IE University, BDBA2025**

**Team:** _[add names]_  
**Dataset:** FiveThirtyEight Bechdel data joined to IMDb/TMDb metadata (Kaggle *9000+ Movies: IMDb and Bechdel*)

---

## 1. Problem statement

The [Bechdel test](https://bechdeltest.com/) asks three questions of a film: (1) does it have at least two named women, (2) who talk to each other, (3) about something other than a man? It is a famously **coarse, contestable, and low-bar** measure of female representation. A film can pass and still be misogynistic; it can fail and still be a landmark of feminist cinema. That coarseness is exactly why it is an interesting ML target: the label is externally defined, community-curated, and disagreements with it are generative rather than errors.

We frame the task as **binary classification**: given pre-release-observable metadata (year, runtime, genre, budget, and production-scale proxies), can we predict whether a film will pass the Bechdel test?

Our central question is **not** 'how high can we push AUC?' Instead, we ask two connected questions:

1. **How much signal does metadata carry about Bechdel outcome?**
2. **Is the model learning something meaningful about representation, or is it taking a shortcut through genre and era?**

The second question motivates a **genre-ablation experiment** (§8.3): we train a version of our best model with all genre features removed and compare its performance to the full model. If most of the AUC survives genre removal, the model is picking up on broader patterns; if AUC collapses, we have learned that our predictor is essentially a genre classifier dressed up in more features.

## 2. Setup

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_synthetic_data, load_real_data
from preprocessing import (
    engineer_features, build_preprocessor,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
)
from models import make_model, MODEL_NAMES, PARAM_GRIDS
from evaluation import (
    cv_score_model, evaluate_on_test, results_table,
    plot_confusion, plot_roc, plot_learning_curve,
)

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

## 3. Load data

The pipeline loads from a local CSV if available and otherwise falls back to a synthetic generator. The synthetic fallback exists only so the pipeline can be developed before the Kaggle CSV is in hand — **metrics from synthetic data are not scientific results, and every plot produced from synthetic data is watermarked below.**

In [2]:
REAL_DATA_PATH = '../data/movies_bechdel.csv'

if os.path.exists(REAL_DATA_PATH):
    df = load_real_data(REAL_DATA_PATH)
    DATA_SOURCE = 'real'
    print(f'Loaded REAL data from {REAL_DATA_PATH}')
else:
    df = load_synthetic_data(n=9000, seed=RANDOM_STATE)
    DATA_SOURCE = 'synthetic'
    print('[!] Real data not found — using synthetic data for pipeline development.')
    print('    Place movies_bechdel.csv in data/ and rerun before reporting results.')

# Helper: stamp plots when running on synthetic data so no one accidentally
# uses a dev-mode figure in the poster or report.
def stamp_if_synthetic(fig):
    if DATA_SOURCE != 'synthetic':
        return
    fig.text(0.5, 0.5, 'SYNTHETIC DATA', fontsize=40, color='red',
             alpha=0.18, ha='center', va='center', rotation=30, zorder=10)

print(f'\nShape: {df.shape}')
df.head()

ValueError: Loaded CSV is missing expected columns: ['imdb_id', 'budget', 'revenue', 'imdb_rating', 'genres', 'bechdel_score', 'bechdel_pass']. You may need to extend the rename_map in load_real_data().

## 4. Exploratory data analysis

### 4.1 Target distribution

In [ ]:
pass_rate = df['bechdel_pass'].mean()
n_pass = int(df['bechdel_pass'].sum())
n_fail = len(df) - n_pass
print(f'Bechdel pass rate: {pass_rate:.1%}  ({n_pass:,} pass / {n_fail:,} fail)')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df['bechdel_pass'].value_counts().plot.bar(ax=axes[0], color=['#d62728', '#2ca02c'])
axes[0].set_xticklabels(['Fail', 'Pass'], rotation=0)
axes[0].set_title('Binary target: Bechdel pass/fail')
axes[0].set_ylabel('Count')
df['bechdel_score'].value_counts().sort_index().plot.bar(ax=axes[1], color='#1f77b4')
axes[1].set_title('Ordinal score (0 = fails first criterion ... 3 = passes all)')
axes[1].set_xlabel('Score')
stamp_if_synthetic(fig)
plt.tight_layout()

The class balance is roughly 55/45, so accuracy is a *usable* metric but we still prioritise F1 and ROC-AUC, which reward correct minority-class prediction rather than simple majority voting.

### 4.2 Missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print('Missingness (fraction of rows):')
print(miss.round(3))

fig, ax = plt.subplots(figsize=(7, 3))
miss.plot.barh(ax=ax, color='#ff7f0e')
ax.set_xlabel('Fraction missing')
ax.set_title('Missingness by column')
ax.invert_yaxis()
stamp_if_synthetic(fig)
plt.tight_layout()

**Interpretation.** Budget and revenue are the most incomplete columns. This is expected: TMDb and IMDb systematically under-report financials for indie and older films, which means a missing budget is **not missing-at-random** — it correlates with production scale, which itself plausibly correlates with the target. Rather than drop these rows, we **median-impute inside the pipeline** (so imputation statistics are learned on each CV fold's training slice, never on held-out data) and keep the information. A natural extension would be to add a binary `budget_is_missing` indicator as a feature, since the missingness pattern itself is informative.

### 4.3 Numeric distributions by target

In [ ]:
df_eda = engineer_features(df)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), ['year', 'runtime', 'log_budget', 'imdb_rating']):
    for val, label, color in [(0, 'Fail', '#d62728'), (1, 'Pass', '#2ca02c')]:
        subset = df_eda.loc[df_eda['bechdel_pass'] == val, col].dropna()
        ax.hist(subset, bins=30, alpha=0.55, label=label, color=color, density=True)
    ax.set_title(f'{col}  (by Bechdel outcome)')
    ax.legend()
stamp_if_synthetic(fig)
plt.tight_layout()

### 4.4 Pass rate by genre

This figure is the **central empirical story** of the EDA. If genre effects are strong and directional, the model has real signal — but we will need to check in §8.3 whether that signal is the *only* thing the model uses.

In [ ]:
genre_cols = [c for c in df_eda.columns if c.startswith('genre_')]
pass_rate_by_genre = (
    df_eda[genre_cols + ['bechdel_pass']]
    .melt(id_vars='bechdel_pass', var_name='genre', value_name='in_genre')
    .query('in_genre == 1')
    .groupby('genre')['bechdel_pass']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
)
pass_rate_by_genre.columns = ['pass_rate', 'n_films']
pass_rate_by_genre['genre'] = pass_rate_by_genre.index.str.replace('genre_', '')

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(pass_rate_by_genre['genre'], pass_rate_by_genre['pass_rate'],
        color=plt.cm.RdYlGn(pass_rate_by_genre['pass_rate']))
ax.axvline(pass_rate, color='black', linestyle='--', label=f'Overall ({pass_rate:.0%})')
ax.set_xlabel('Bechdel pass rate')
ax.set_title('Pass rate by genre')
ax.invert_yaxis()
ax.legend()
stamp_if_synthetic(fig)
plt.tight_layout()

pass_rate_by_genre[['n_films', 'pass_rate']].round(3)